# Lakeshore Molded Products: what should press 3 make next?

This notebook is for the planner, the plant manager and the owner. You do not need to
write code. Every gray box below is either a command you can run, or a file you can read.
Run the boxes from top to bottom. The answers are already saved under each box.

## (a) Your business

Lakeshore Molded Products makes plastic storage bins near Muskegon, Michigan. Press 3
makes 12 bins: four sizes (S, M, L, XL) in three colors (grey, black and natural).
Stores order bins every day. If a bin is out of stock, the store buys from someone else.
That sale is lost.

Each morning the planner picks one bin to run that day, or rests the press.

- A new mold in the same color costs about $200 and 10% of the day.
- A new color costs about $1,200 (purge resin, scrap, cleanup) and half of the day.
- Bins on the shelf cost money too: about 1% of their cost each night.
- The biggest customer stocks three bins (GRY-S, GRY-M and BLK-S). Its contract fines
  Lakeshore $3 for every unit it cannot ship of those three.

Today the planner runs the bin that will run out first. That sounds safe. But it
changes color almost every day. **The question: is there a better rule, and how much
money is it worth?**

## (b) Your inputs

The folder holds the files a planner already has. Nothing here is made up by the tool.

| File | What it is | Where it comes from |
|---|---|---|
| `data/products.csv` | The item master: one row per bin | ERP item master export |
| `data/sales-history.csv` | Units sold per bin per day, last 13 weeks | ERP sales export |
| `lakeshore.twin.yaml` | The twin capsule: ties the files together, holds costs and run settings | You write it once |
| `policies/`, `rewards/`, `demand/`, `candidates/` | Your rules, as short files | See part (d) |

### The item master

In [1]:
!cat data/products.csv

id,family,abc,demand,run_rate,price,unit_cost,max_storage
GRY-S,grey,A,60.5,450,12.00,7.20,1500
GRY-M,grey,A,39.7,360,18.00,10.80,1000
GRY-L,grey,B,18.2,270,26.00,15.60,540
GRY-XL,grey,C,5.6,180,38.00,22.80,240
BLK-S,black,A,44.8,450,12.60,7.56,1125
BLK-M,black,B,21.8,360,18.90,11.34,660
BLK-L,black,C,8.5,270,27.30,16.38,360
BLK-XL,black,C,5.5,180,39.90,23.94,200
NAT-S,natural,B,19.9,450,13.20,7.92,600
NAT-M,natural,B,15.7,360,19.80,11.88,480
NAT-L,natural,C,8.0,270,28.60,17.16,320
NAT-XL,natural,C,3.9,180,41.80,25.08,160


What each column means:

| Column | Meaning |
|---|---|
| `id` | The bin's part number. Grey is GRY, black is BLK, natural is NAT. |
| `family` | The color. Changing color is the expensive change. |
| `abc` | A for fast sellers, B for middle, C for slow. |
| `demand` | Average units sold per day. It is the average of the sales history below. |
| `run_rate` | Units the press makes in a full day of this bin. |
| `price` | What a store pays for one bin, in dollars. |
| `unit_cost` | What one bin costs to make, in dollars (resin, labor, power). |
| `max_storage` | The most units of this bin the warehouse can hold. |

### The sales history

The file has one row per day and bin: 91 days times 12 bins. Here are the first rows.

In [2]:
!head -13 data/sales-history.csv

date,product,units_sold
2026-08-17,GRY-S,66
2026-08-17,GRY-M,44
2026-08-17,GRY-L,21
2026-08-17,GRY-XL,5
2026-08-17,BLK-S,49
2026-08-17,BLK-M,18
2026-08-17,BLK-L,9
2026-08-17,BLK-XL,8
2026-08-17,NAT-S,21
2026-08-17,NAT-M,26
2026-08-17,NAT-L,11
2026-08-17,NAT-XL,5


In [3]:
!wc -l data/sales-history.csv

    1093 data/sales-history.csv


| Column | Meaning |
|---|---|
| `date` | The day of the sale. |
| `product` | The bin's part number. It must match an `id` in the item master. |
| `units_sold` | Units shipped to stores that day. |

### The twin capsule

The capsule says which files to read, what a change costs, and how to run the tests.
The line `products: data/products.csv` reads the item master from its own file.

In [4]:
!cat lakeshore.twin.yaml

# Lakeshore Molded Products: the twin capsule for press 3.
# It ties the input files together and holds the money rules and the run settings.
schema_version: '0.1'
required_capabilities:
  - make_to_stock.step
model:
  domain: make_to_stock
  revision: '1'
  period: day
  horizon: 60                 # plan 60 days ahead
  start_cover_days: 8         # each bin starts with about 8 days of stock
  line:
    id: press-3
    changeover:               # what a mold or color change costs
      within_family: {cost: 200, output_lost: 0.10}    # new mold, same color: $200 and 10% of the day
      across_family: {cost: 1200, output_lost: 0.50}   # new color: $1,200 purge and half the day
  products: data/products.csv # the item master: one row per bin
  economics:
    defaults: {holding_rate: 0.01, lost_sale_rate: 0.30}   # stock costs 1% of unit cost a night; a lost sale costs 30% of price
  state_groups:
    views:
      simple:                 # the state map label: stock level per color, an

## (c) Check and run

First, check the files. A check finds a typo or a missing number before you run anything.
No message from the first command means the capsule is good.

In [5]:
!twinflow scenario validate lakeshore.twin.yaml

In [6]:
!twinflow extensions check .

No problems found.


These are the rules the race will test. `project` rules are files in your `policies/`
folder. `builtin` rules come with twinflow.

In [7]:
!twinflow policies list .

name                source   decision   settings                                 problems
abc_wheel           builtin  make_next  days_per_slot                            0
dollar_shortfall    builtin  make_next  target_days                              0
family_campaign     project  make_next  target_days                              0
fixed_wheel         builtin  make_next  days_per_slot                            0
idle                builtin  make_next  -                                        0
kanban              builtin  make_next  target_days, trigger_share               0
lowest_cover        builtin  make_next  stop_days                                0
lowest_stock_first  project  make_next  stop_days                                0
myopic              builtin  make_next  -                                        0
random              builtin  make_next  -                                        0
s_big_s             builtin  make_next  safety_days, eoq_scale                  

### A quick run of today's rule

This runs the planner's rule (`lowest_cover`: run the bin that runs out first) for 20
runs of 60 days. This quick run draws sales at random around each bin's average.

In [8]:
!twinflow scenario run lakeshore.twin.yaml --out runs/today --summary

Ran 20 runs (seed 2026).
Profit per run:  $41,168
Fill rate:       93.7% of demand shipped
Changeovers:     60.0 per run, 41.7 of them to a new family
Money per run:   sales $253,893 | making $144,361 | setups $53,700 | holding $9,620 | lost sales $5,044
Full result: runs/today/result.json


Look at the changeovers: about 60 changes in 60 days, and most are color changes.
Setups cost more than $50,000 per run.

### The race

The race tests every rule on the same days of real sales. Each run picks 60 random days
from your sales history (`demand/sales_history.py`). Every rule gets the same picks, so a
win is not luck. The score is dollars less the customer's fines
(`rewards/protect_key_items.py`). It takes about 15 seconds.

In [9]:
!twinflow policies race . --out runs/race

Racing on 40 runs per policy (seed 7).
  abc_wheel (full): 35,161.80
  dollar_shortfall (full): 31,962.45
  family_campaign (full): 63,690.08
  fixed_wheel (full): 23,952.02
  idle (full): -57,262.35
  kanban (full): 38,834.04
  lowest_cover (full): 36,961.20
  lowest_stock_first (full): 36,961.20
  myopic (full): 14,006.90
  random (full): -35,530.12
  s_big_s (full): 55,678.64
  setup_aware_runout (full): 47,873.45

Policies ranked by protect_key_items (dollars are always shown), under the full view.
Rank  Policy              Score    Dollars per run                          Fill rate  Changeovers
1     family_campaign     63,690   $63,859 (95% range 62,177 to 65,540)     96.8%      44.9
2     s_big_s             55,679   $56,397 (95% range 54,603 to 58,191)     94.0%      28.6
3     setup_aware_runout  47,873   $48,302 (95% range 46,635 to 49,968)     98.4%      58.8
4     kanban              38,834   $40,712 (95% range 38,619 to 42,805)     92.5%      58.8
5     lowest_cover       

How to read it:

- **Score** is the number the race sorts by: dollars after the customer's fines.
- **Dollars per run** is the profit of 60 days, with a 95% range. If two ranges do not
  overlap, the gap is real.
- **Fill rate** is the share of store orders you shipped.
- **Changeovers** is how many times the press changed bins in 60 days.

### The state map

The race keeps the days of the best rules. Each day gets a short label: the stock level
of each color (out, low, ok or high), then the color in the press. This shows how the
winning rule moves the plant from day to day.

In [10]:
!twinflow states runs/race/policies/family_campaign --view simple --top 5

Moves found in runs/race/policies/family_campaign
Runs found: 1
Episodes: 40
Moves: 2400
Time span: 0 to 60
Unique states: 2393
Top states (view simple), share of moves:
    8.2%  grey:high black:high natural:high | grey
    6.9%  grey:ok black:ok natural:ok | grey
    6.8%  grey:high black:high natural:high | black
    6.0%  grey:high black:ok natural:high | grey
    6.0%  grey:high black:high natural:high | natural
Ledger by part (all moves):
  revenue     10,392,445.30
  production   6,089,158.20
  setup          691,400.00
  holding        934,971.50
  lost_sales     122,574.54
  late                 0.00
  total        2,554,341.06


In [11]:
!twinflow markov runs/race/policies/family_campaign --view simple --top 6 --min-prob 0.1

View: simple
Moves: 2400
Coverage: 39.4% of moves start in the 6 states below
States:
    8.2%  grey:high black:high natural:high | grey  (sticky)
    6.9%  grey:ok black:ok natural:ok | grey  (sticky)
    6.8%  grey:high black:high natural:high | black  (sticky)
    6.0%  grey:high black:ok natural:high | grey  (sticky)
    6.0%  grey:high black:high natural:high | natural  (sticky)
    5.5%  grey:ok black:high natural:high | grey
Likely moves (8):
   0.71  grey:ok black:ok natural:ok | grey  ->  grey:ok black:ok natural:ok | grey
   0.70  grey:high black:high natural:high | black  ->  grey:high black:high natural:high | black
   0.63  grey:high black:ok natural:high | grey  ->  grey:high black:ok natural:high | grey
   0.63  grey:high black:high natural:high | grey  ->  grey:high black:high natural:high | grey
   0.62  grey:high black:high natural:high | natural  ->  grey:high black:high natural:high | natural
   0.49  grey:ok black:high natural:high | grey  ->  grey:ok black:high na

"Sticky" means the plant tends to stay in that state. The winning rule stays in one
color for several days. It does not bounce between colors.

### Try a new rule

The production manager has an idea: run in color campaigns, but never let a key bin of
the big customer run out. That rule is the file `candidates/campaign_guard_key_items.py`.
`improve` races it against your current rules. It joins `policies/` only if it wins and
the win is proven (the 95% range of the gain is above zero).

In [12]:
!twinflow improve . --candidate candidates/campaign_guard_key_items.py --out runs/improve

Racing campaign_guard_key_items against 2 of the policies in this project on 40 runs each (seed 7).
  campaign_guard_key_items (full): 66,277.72
  family_campaign (full): 63,690.08
  lowest_stock_first (full): 36,961.20

Candidate campaign_guard_key_items: score 66,277.72, dollars $66,278 per run (settings: target_days=16, alarm_days=2.0).
Previous best family_campaign: score 63,690.08, dollars $63,859 per run (settings: target_days=16).
Candidate minus previous best, same draws: score +2,587.65 (95% range +1,204.98 to +3,970.31)
Candidate minus previous best, same draws: dollars +$2,419.20 (95% range +1,033.79 to +3,804.60)
Won. Copied candidates/campaign_guard_key_items.py to ./policies/campaign_guard_key_items.py.
Proof: runs/improve/improve.json
Page: runs/improve/improve.html


## (d) Your rules

Each rule is a short file in the folder. A file is shown below in a box that starts
with `%%writefile`. That box saves the file, so you can change a line and run the box
again. Lines marked `<- CHANGE` are the ones a planner changes most. Lines marked
`KEY LINE` are where the rule makes its choice.

### `policies/lowest_stock_first.py`: today's rule, written out

This is the rule the planner uses now. It is the same as the built-in `lowest_cover`.
It is here so you can read it and copy it.

In [13]:
%%writefile policies/lowest_stock_first.py
"""Make the bin that will run out first.

This is a policy file: it picks what press 3 makes each day. It is the planner's rule
today. Look at every bin and count how many days its stock will last at the usual sales
rate. Run the one with the fewest days left. If every bin has more than `stop_days` days
of stock, rest the press.
"""

NAME = "lowest_stock_first"

# Values the race may try. <- CHANGE: when every bin has this many days of stock, rest.
SETTINGS = {"stop_days": [6, 10, 15, 25]}


def decide(state, stop_days=1000):
    # Start with the first bin, then look for one that runs out sooner.
    neediest = state.products[0]
    for product in state.products:
        if state.days_of_cover[product] < state.days_of_cover[neediest]:
            neediest = product

    # Plenty of stock everywhere: rest the press and save the setup cost.
    if state.days_of_cover[neediest] >= stop_days:
        return "idle"
    # KEY LINE: run the bin that runs out first, even if it is a new color.
    return neediest

Overwriting policies/lowest_stock_first.py


### `policies/family_campaign.py`: stay in one color

This rule stays in the color the press is set up for until every bin of that color has
`target_days` of stock. Then it moves to the color that will run out first. The race
tries each value in `SETTINGS` and keeps the best one.

In [14]:
%%writefile policies/family_campaign.py
"""Stay with one color until it is stocked up, then move on.

This is a policy file: it picks what press 3 makes each day. A new mold in the same
color is cheap. A new color costs a lot. So the press stays in its current color until
every bin of that color has `target_days` days of stock. Then it moves to the color of
the bin that will run out first.
"""

import math

NAME = "family_campaign"

# Values the race may try. <- CHANGE: the days of stock to build before leaving a color.
SETTINGS = {"target_days": [6, 8, 12, 16, 20]}


def decide(state, target_days=8):
    # Find the bins that are below their stock goal.
    short = []
    for product in state.products:
        # Do not stock more than the warehouse holds, or more than the days left can sell.
        days = min(target_days, state.days_left)
        goal = min(state.max_storage[product], math.ceil(days * state.demand_rate[product]))
        if state.stock[product] < goal:
            short.append(product)

    # Everything is stocked up: rest the press.
    if not short:
        return "idle"

    # KEY LINE: prefer bins in the color the press is already set up for.
    current_family = state.family[state.setup]
    same_family = [p for p in short if state.family[p] == current_family]
    choices = same_family or short

    # Among those, run the one that will run out first.
    neediest = choices[0]
    for product in choices:
        if state.days_of_cover[product] < state.days_of_cover[neediest]:
            neediest = product
    return neediest

Overwriting policies/family_campaign.py


### `candidates/campaign_guard_key_items.py`: the new idea

This is the family campaign with one guard: an A bin with less than `alarm_days` of stock
goes first, even when that means a color change. It sits in `candidates/` until
`improve` proves it wins.

In [15]:
%%writefile candidates/campaign_guard_key_items.py
"""Stay in one color until it is stocked up, but never let a key bin run out.

This is a candidate policy: a new rule to test against the ones we use. It starts
from `family_campaign`. One change: if a key bin (an A item) is about to run out,
the press switches to it first, even when that means a color change.
"""

import math

NAME = "campaign_guard_key_items"

# Values the race may try. <- CHANGE: the stock goal in days, and the alarm level.
SETTINGS = {"target_days": [8, 12, 16], "alarm_days": [1.0, 2.0]}


def decide(state, target_days=12, alarm_days=2.0):
    # KEY LINE: an A item with less than `alarm_days` of stock goes first, color change or not.
    urgent = [p for p in state.products if state.abc[p] == "A" and state.days_of_cover[p] < alarm_days]
    if urgent:
        return min(urgent, key=lambda p: state.days_of_cover[p])

    # Otherwise, the family campaign: find the bins below their stock goal.
    short = []
    for product in state.products:
        days = min(target_days, state.days_left)
        goal = min(state.max_storage[product], math.ceil(days * state.demand_rate[product]))
        if state.stock[product] < goal:
            short.append(product)
    if not short:
        return "idle"

    # Stay in the color the press is set up for, when that color still needs stock.
    current_family = state.family[state.setup]
    same_family = [p for p in short if state.family[p] == current_family]
    choices = same_family or short
    return min(choices, key=lambda p: state.days_of_cover[p])

Overwriting candidates/campaign_guard_key_items.py


### `rewards/protect_key_items.py`: how a day is scored

The race ranks rules by this score. It is the day's dollars, less the contract fine for
each key bin you could not ship. The capsule picks it with `score: protect_key_items`.

In [16]:
%%writefile rewards/protect_key_items.py
"""Score each day by its dollars, less a fine for every key bin we could not ship.

This is a reward file. Our largest customer stocks three bins on its shelves. Its
contract fines us for each unit we cannot ship of those bins. The fine is on top of
the lost sale itself, so the race should rank rules by dollars after fines.
"""

NAME = "protect_key_items"

# <- CHANGE: the bins in the customer's contract, and the fine per unit short, in dollars.
SETTINGS = {"key_items": ["GRY-S", "GRY-M", "BLK-S"], "fine_per_unit": 3.0}


def reward(move, key_items=(), fine_per_unit=0.0):
    # Units asked for but not shipped show up as events named "lost.<bin>".
    units_short = 0.0
    for name, units in move.events.items():
        if name.startswith("lost.") and name[len("lost."):] in key_items:
            units_short += units
    # KEY LINE: the day's dollars (sales less every cost) less the contract fine.
    return move.ledger.total - fine_per_unit * units_short

Overwriting rewards/protect_key_items.py


### `demand/sales_history.py`: where tomorrow's sales come from

This file reads `data/sales-history.csv` and, for each simulated day, picks a real past
day at random. The capsule picks it with `demand: sales_history`.

In [17]:
%%writefile demand/sales_history.py
"""Tomorrow's sales look like a real day from the last 13 weeks.

This is a demand file. It reads `data/sales-history.csv` (one row per day and bin,
exported from the ERP). For each simulated day it picks one real past day at random
and uses that day's sales for every bin. Busy Mondays and slow Sundays stay in the mix,
and so do the days when several bins sold well together.
"""

import csv
from pathlib import Path

NAME = "sales_history"

# The sales file, next to this folder. <- CHANGE: point at a newer export here.
HISTORY_FILE = Path(__file__).resolve().parent.parent / "data" / "sales-history.csv"


def _read_history():
    """Each past day as {bin: units sold}, oldest day first."""
    days = {}
    with HISTORY_FILE.open(newline="") as handle:
        for row in csv.DictReader(handle):
            day = days.setdefault(row["date"], {})
            day[row["product"]] = int(row["units_sold"])
    return [days[date] for date in sorted(days)]


HISTORY = _read_history()


def demand(period, rng, model):
    # KEY LINE: pick one real past day at random. The race gives every rule the same picks.
    day = HISTORY[int(rng.integers(len(HISTORY)))]
    sales = {}
    for product in model.products:
        if product in day:
            sales[product] = day[product]
        else:
            # A new bin with no history yet: draw around its average from products.csv.
            sales[product] = int(rng.poisson(model.demand_rate[product]))
    return sales

Overwriting demand/sales_history.py


## (e) Results and the decision

What the commands said, for 60 days of press 3:

| Rule | Profit per 60 days | Fill rate | Changeovers |
|---|---|---|---|
| Today: run what runs out first (`lowest_cover`) | about $39,100 | 91% | 58 |
| Stay in one color (`family_campaign`, 16-day goal) | about $63,900 | 97% | 45 |
| Stay in one color, guard the key bins (new, from `improve`) | about $66,300 | | |

**The recommendation:** switch press 3 to color campaigns with the key-bin guard.

- It earns about $27,000 more per 60 days than today's rule. That is about $165,000 a year.
- Most of the gain is fewer color changes. Fewer purges also means less scrap resin.
- The guard adds about $2,400 per 60 days on top of the plain campaign, because the big
  customer's bins do not run out. The 95% range of that gain is above zero, so it is real.
- The press holds more stock under this rule. Check that the warehouse can hold it.
  The `max_storage` column already limits each bin.

Open the full race report in a web browser: [runs/race/race.html](runs/race/race.html).
It shows the winner, the money by cause, and the state map. The proof of the new rule is
in [runs/improve/improve.json](runs/improve/improve.json). Both are made when you run
the notebook.

## (f) Try it yourself

1. **A bigger fine.** In `rewards/protect_key_items.py`, change `fine_per_unit` from
   `3.0` to `10.0`. Run the race box again. Does the ranking change?
2. **A new color cost.** In `lakeshore.twin.yaml`, change the `across_family` cost from
   `1200` to `600` (a faster purge compound). Run the race again. How much does the
   campaign rule still win by?
3. **Newer sales.** Replace `data/sales-history.csv` with your newest export (same three
   columns). Update the `demand` column of `data/products.csv` to the new averages. Run
   the check, then the race.

To start over, delete `policies/campaign_guard_key_items.py` (the `improve` box copies
it there when it wins) and the `runs/` folder.

## For analysts (optional)

Two longer notebooks rebuild a research project's lessons with twinflow's Python API:
[../../molding-press/notebooks/analyst/](../../molding-press/notebooks/analyst/). They are
for analysts who want charts, tuning tables and Markov heat maps. A planner does not need them.